# Read one batch with PySpark

## Setup
Use Python (clickstream), PySpark 4.0.1, and Java 17 on Windows. Run from `data/notebooks/`. Run notebook 01 first to generate `../batches/clickstream_2008-04-01.csv`. No variables are shared between notebooks. The initial environment checks may report no Java until the configuration cells run.

Saved outputs were retained from earlier runs. Restart the kernel and Run All to validate this standalone notebook.

## Steps

### Inspect the environment

In [1]:
# Report installed versions and Java visibility before configuring this kernel.
import sys
import shutil
import subprocess
from importlib.metadata import version, PackageNotFoundError

print("Python version:", sys.version)
print("Python executable:", sys.executable)

try:
    print("PySpark version:", version("pyspark"))
except PackageNotFoundError:
    print("PySpark is not installed in this environment.")

java_path = shutil.which("java")
print("Java executable:", java_path)

if java_path:
    result = subprocess.run(
        [java_path, "-version"],
        capture_output=True,
        text=True
    )
    print(result.stdout)
    print(result.stderr)

Python version: 3.11.17 | packaged by conda-forge | (main, Oct 13 2026, 16:36:10) [MSC v.1944 64 bit (AMD64)]
Python executable: c:\conda\envs\clickstream\python.exe
PySpark version: 4.0.1
Java executable: None


### Check the Python kernel

In [2]:
# Confirm that the clickstream environment is selected.
import sys
import os
import shutil
import pyspark

print("Python executable:", sys.executable)
print("PySpark version:", pyspark.__version__)
print("Java executable:", shutil.which("java"))
print("JAVA_HOME:", os.environ.get("JAVA_HOME"))

Python executable: c:\conda\envs\clickstream\python.exe
PySpark version: 4.0.1
Java executable: None
JAVA_HOME: None


### Locate Conda Java

In [3]:
# Use the OpenJDK installation inside the active Conda environment on Windows.
from pathlib import Path
import sys

java_home = Path(sys.prefix) / "Library"

print("Candidate JAVA_HOME:", java_home)
print("Java exists:", (java_home / "bin" / "java.exe").exists())

Candidate JAVA_HOME: c:\conda\envs\clickstream\Library
Java exists: True


### Configure Java

In [4]:
# Set Java paths for this process. Rerun this cell after restarting the kernel.
import os
import shutil

os.environ["JAVA_HOME"] = str(java_home)
os.environ["PATH"] = (
    str(java_home / "bin") + os.pathsep + os.environ.get("PATH", "")
)

print("JAVA_HOME:", os.environ["JAVA_HOME"])
print("Java executable:", shutil.which("java"))

JAVA_HOME: c:\conda\envs\clickstream\Library
Java executable: c:\conda\envs\clickstream\Library\bin\java.EXE


### Start local Spark

In [5]:
# Use two local threads and two shuffle partitions for this small example.
import sys
import os
from pyspark.sql import SparkSession

# Use the current environment for Python workers
os.environ["PYSPARK_PYTHON"] = sys.executable

spark = (
    SparkSession.builder
    .appName("clickstream-pipeline")
    .master("local[2]")
    .config("spark.sql.shuffle.partitions", "2")
    .getOrCreate()
)

print("Spark version:", spark.version)
print("Spark master:", spark.sparkContext.master)

print("Test row count:", spark.range(10).count())

Spark version: 4.0.1
Spark master: local[2]
Test row count: 10


### Define the input schema

In [6]:
# Match the CSV column order. True allows nulls; quality checks must handle them.
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

schema = StructType([
    StructField("year", IntegerType(), True),
    StructField("month", IntegerType(), True),
    StructField("day", IntegerType(), True),
    StructField("order", IntegerType(), True),
    StructField("country", IntegerType(), True),
    StructField("session ID", IntegerType(), True),
    StructField("page 1 (main category)", IntegerType(), True),
    StructField("page 2 (clothing model)", StringType(), True),
    StructField("colour", IntegerType(), True),
    StructField("location", IntegerType(), True),
    StructField("model photography", IntegerType(), True),
    StructField("price", IntegerType(), True),
    StructField("price 2", IntegerType(), True),
    StructField("page", IntegerType(), True),
])

### Read a batch with Spark

In [7]:
# Validate the header and fail on malformed input. Business checks come later.
from pathlib import Path

batch_path = Path("../batches/clickstream_2008-04-01.csv").resolve()

clicks = (
    spark.read
    .option("header", True)
    .option("sep", ";")
    .option("mode", "FAILFAST")
    .option("enforceSchema", False)
    .schema(schema)
    .csv(str(batch_path))
)

clicks.printSchema()
clicks.show(5, truncate=False)

print("Batch row count:", clicks.count())

root
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- day: integer (nullable = true)
 |-- order: integer (nullable = true)
 |-- country: integer (nullable = true)
 |-- session ID: integer (nullable = true)
 |-- page 1 (main category): integer (nullable = true)
 |-- page 2 (clothing model): string (nullable = true)
 |-- colour: integer (nullable = true)
 |-- location: integer (nullable = true)
 |-- model photography: integer (nullable = true)
 |-- price: integer (nullable = true)
 |-- price 2: integer (nullable = true)
 |-- page: integer (nullable = true)

+----+-----+---+-----+-------+----------+----------------------+-----------------------+------+--------+-----------------+-----+-------+----+
|year|month|day|order|country|session ID|page 1 (main category)|page 2 (clothing model)|colour|location|model photography|price|price 2|page|
+----+-----+---+-----+-------+----------+----------------------+-----------------------+------+--------+-----------------+-

## Checks and next steps
Expected: Spark 4.0.1, local[2], test count 10, and 3,181 batch rows with 14 columns. Next, add cleaning and quality rules.